# 01 · CaRS-50 — build the pool

*Swales CARS moves in research-article introductions (3 or 11 classes)*

### Where this sits

```
▶ 01 build the pool  →  02 sample  →  03 annotate  →  04 prompt  →  05 report
```

You run **01 once per group**, for your own track only. It ends by writing `data/pools/<track>_pool.json` — the file notebook 02 opens.

---

**What it is.** 50 BioRxiv article introductions, annotated sentence by sentence with Swales' CARS Move and Step scheme. **The annotators themselves reached only κ ≈ 0.43** — so on this track, "the model is wrong" and "the scheme is fuzzy" are both live explanations, and telling them apart is the interesting part.

**Difficulty of the labeling judgment:** ★★★ — hard. Judging moves in an introduction needs more context than a single sentence gives you.

**Licence:** CC BY 4.0  
**Cite:** Lam, C. & Nnamoko, N. (2025). *Mendeley Data*, V1. doi:10.17632/kwr9s5c4nk.1

---

Every dataset in this course is reshaped into the **same canonical schema**, so one pipeline works for all of them:

```json
[{"id": 1, "text": "...", "label": "..."}]
```

The *raw* data, though, looks different every time. **That difference is the lesson** — half of building a gold standard is getting messy real data into a clean, consistent shape.

> The reshaping code below is read straight out of `scripts/reshape.py` — it is the same code `scripts/prep_datasets.py` runs, not a copy of it. What is *missing* from it is missing on purpose: the ✏️ cells are the decisions, and they are yours. (Generated by `scripts/_generate_pool_notebooks.py`; edit that or `reshape.py`, never the `.ipynb`.)

## Step 1 — Download the raw data

This one is on **Mendeley Data**, which has a public API. We ask it for the dataset's file list, then download each file. The CDN refuses requests that do not look like a browser, hence the `User-Agent` header.

In [ ]:
import json, urllib.request, pathlib

RAW_DIR = pathlib.Path("cars50")
RAW_DIR.mkdir(exist_ok=True)

def fetch(url):
    request = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
    return urllib.request.urlopen(request, timeout=60)

meta = json.loads(fetch("https://data.mendeley.com/public-api/datasets/kwr9s5c4nk").read())
for record in meta["files"]:
    target = RAW_DIR / record["filename"]
    if not target.exists():
        target.write_bytes(fetch(record["content_details"]["download_url"]).read())
print("downloaded", len(list(RAW_DIR.glob("*.xml"))), "XML files")

## Step 2 — Look at the raw format

**XML** this time. Each sentence carries a `step` code like `1b`:

```xml
<sentence><sentenceID/><text/><step>1b</step></sentence>
```

In [ ]:
print(open(sorted(RAW_DIR.glob("*.xml"))[0], encoding="utf-8").read()[:900])

## Step 3 — Reshape into the canonical schema

The parsing is written for you, and it gives you **both granularities at once**:

- the leading digit of `1b` is the **Move** → 3 classes;
- the whole code `1b` is the **Step** → 11 classes.

Sentences with no code, or a code that does not start with a move digit, are dropped either way.

✏️ **Which one you study is the decision**, and on this track it is the whole shape of the project. Three classes with a few hundred items each is a fair task you can sample 40 items from comfortably. Eleven classes over the same sentences means some steps have barely a dozen examples, a confusion matrix with 121 cells, and an annotation job your two coders will find genuinely hard — remember the original annotators managed only κ ≈ 0.43 at this granularity.

Neither is the safe answer. The 11-class version makes a better project **if** you have the time to annotate it properly and the nerve to report a low F1 with a good explanation. Decide now, write it in `PLAN.md`, and do not switch after you have seen the numbers.

In [ ]:
import xml.etree.ElementTree as ET
from pathlib import Path

def reid(items):
    """Renumber ids sequentially from 1, keeping the current order."""
    renumbered = []
    next_id = 1
    for item in items:
        new_item = dict(item)
        new_item["id"] = next_id
        renumbered.append(new_item)
        next_id = next_id + 1
    return renumbered

def reshape_cars50(cars50_dir):
    """Parse the 50 XML introductions into TWO datasets: moves, and move+step.

    XML shape:
        <sentence><sentenceID/><text/><step>1b</step></sentence>

    The `step` code is like "1b": the leading DIGIT is the Move, the whole code is the
    Step. So one parse gives two granularities, and which you use is a scheme decision:
    3 classes is a fair task, 11 classes is the stretch version. Returns
    (move_rows, step_rows).
    """
    source_dir = Path(cars50_dir)
    move_rows = []
    step_rows = []
    for xml_path in sorted(source_dir.glob("*.xml")):
        tree = ET.parse(xml_path)
        for sentence in tree.iter("sentence"):
            text_element = sentence.find("text")
            step_element = sentence.find("step")
            if text_element is None or step_element is None:
                continue
            text = (text_element.text or "").strip()
            code = (step_element.text or "").strip()
            # Skip anything unlabelled, or whose code does not start with a move digit.
            if not text or not code or not code[0].isdigit():
                continue
            move_rows.append({"id": 0, "text": text, "label": "Move " + code[0]})
            step_rows.append({"id": 0, "text": text, "label": code})
    return reid(move_rows), reid(step_rows)

In [ ]:
move_rows, step_rows = reshape_cars50(RAW_DIR)
print("moves:", len(move_rows), " steps:", len(step_rows))

from collections import Counter
print("move classes:", Counter(r["label"] for r in move_rows))
print("step classes:", Counter(r["label"] for r in step_rows))

In [ ]:
# ✏️ Step 3a · Choose your granularity ───────────────────────────
# Goal      : pick the version of the scheme your group will actually study.
# Shape     : rows = move_rows    # 3 classes
#             rows = step_rows    # 11 classes
# Produce   : rows (a list) — either move_rows or step_rows      ← later cells use this name
# Note      : one line. Spend the time on the ARGUMENT, not the typing —
#             PLAN.md asks you to justify it in a sentence.
# Careful   : whichever you pick, the label names in your prompt and your
#             annotation sheet must match these exactly ("Move 1", or
#             "1b").

# ✏️ your code here


## Step 4 — Check the label balance

In [ ]:
from collections import Counter

print("total items:", len(rows))
print("label counts:", dict(Counter(item["label"] for item in rows)))
rows[:3]        # peek at the first three reshaped items

In [ ]:
# ✏️ Step 4b · React to the balance ──────────────────────────────
# Goal      : decide what the counts you just printed mean for your study.
# Shape     : MIN_PER_CLASS = <the size of your SMALLEST class>
#             that is the ceiling on N_PER_CLASS in config.py — a balanced
#             sample cannot draw more from a class than the class has
# Produce   : MIN_PER_CLASS (an int)      ← later cells use this name
# Note      : if you chose steps, look at how thin the rare ones are before you commit.
# Note      : if the rarest class is tiny, say so in PLAN.md. Merging it
#             away or living with fewer items are both defensible;
#             not noticing is not.

# ✏️ your code here


## Step 5 — Save it

In [ ]:
# Save the pool. Two places you might want it:
#   * this repo, if you cloned it:  "../data/pools/cars50_pool.json"
#   * your Google Drive, so it survives the Colab runtime resetting
import json, pathlib

OUT_FILE = "../data/pools/cars50_pool.json"

# In Colab WITHOUT the repo, uncomment these two to write straight to Drive:
# from google.colab import drive; drive.mount("/content/drive")
# OUT_FILE = "/content/drive/MyDrive/cars50_pool.json"

pathlib.Path(OUT_FILE).parent.mkdir(parents=True, exist_ok=True)
with open(OUT_FILE, "w", encoding="utf-8") as f:
    json.dump(rows, f, ensure_ascii=False, indent=2)
print("Saved", len(rows), "items to", OUT_FILE)

# If you chose steps, save as cars50_step_pool.json and point config.py at that name.

## What you just built, and what happens to it

This is the **pool** — everything usable in the corpus, with its natural label imbalance intact. It is **not** your gold set, and its labels are **not** your labels: they are the original corpus authors' judgment, and you have not yet agreed with them about anything.

What those labels are for is narrow, and worth being precise about:

1. **Stratifying the draw** in notebook 02 — you cannot sample evenly across classes without knowing what the classes are.
2. **A comparison** in notebook 03 — once you have annotated blind and adjudicated, `compare_to_published` shows you every item where your group landed somewhere different. That gap is evidence, and one of the more interesting things you can put in a report.

They are never the answer key you score the model against. That file does not exist yet — you make it in notebook 03.

---

**Next:** set `TRACK = "cars50"` in `config.py`, then open `02_sample.ipynb`.